In [1]:
import numpy as np
import pandas as pd

In [2]:
import xml.etree.ElementTree as ET

tree = ET.parse("../data/raw/export.xml")
root = tree.getroot()

In [8]:
workouts = root.findall("Workout")

rows = []

for workout in workouts:
    row = workout.attrib.copy()
    
    for child in workout:
        tag = child.tag
        attrib = child.attrib
        
        # Handle MetadataEntry elements
        if tag == "MetadataEntry":
            key = attrib.get("key")
            value = attrib.get("value")
            
            if key == "HKElevationAscended":
                row["elevation_ascended"] = value
            elif key == "HKWeatherTemperature":
                row["temperature"] = value
            elif key == "HKWeatherHumidity":
                row["humidity"] = value
        
        # Handle WorkoutStatistics elements
        elif tag == "WorkoutStatistics":
            stat_type = attrib.get("type")
            
            if stat_type == "HKQuantityTypeIdentifierDistanceWalkingRunning":
                row["distance_km"] = attrib.get("sum")
            elif stat_type == "HKQuantityTypeIdentifierBasalEnergyBurned":
                row["basal_calories"] = attrib.get("sum")
            elif stat_type == "HKQuantityTypeIdentifierActiveEnergyBurned":
                row["active_calories"] = attrib.get("sum")
            elif stat_type == "HKQuantityTypeIdentifierHeartRate":
                row["heart_rate_average"] = attrib.get("average")
                row["heart_rate_min"] = attrib.get("minimum")
                row["heart_rate_max"] = attrib.get("maximum")
    
    rows.append(row)

df = pd.DataFrame(rows)

# Filter to only walking workouts
df = df[df["workoutActivityType"] == "HKWorkoutActivityTypeWalking"]

In [10]:
df.to_csv("../data/raw/raw_walks.csv", index=False)